# Offline saved-output QA evaluation

Scores all nine saved model variants, including Gemma, against canonical
validation references joined and checked by ID. Writes JSON provenance, chrF,
Urdu-normalized F1, human counts, and paired article-bootstrap intervals to
`paper/generated`. No model inference or remote metric downloads occur.

The historical mT5 response CSV and human-annotated workbook contain different
predictions and are explicitly reported as separate variants. Frozen historical
METEOR values in the main paper are preserved; recomputation uses one common
documented NLTK implementation. Missing local NLTK resources cause an explicit
error rather than being downloaded silently.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().resolve()
while not (root / "scripts" / "paper_analysis.py").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook inside the LEGAL-UQA checkout")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from scripts.paper_analysis import analyze
results = analyze(root / "paper" / "generated")


In [ ]:
rows = []
for model, result in results["models"].items():
    rows.append({"model": model, **{key: result[key] for key in
        ["f1", "meteor", "sacrebleu", "chrf", "urdu_normalized_f1"]},
        "human_score_percent": result["human_score_percent"] if result["human_same_predictions"] else None})
display(pd.DataFrame(rows))
display(pd.DataFrame(results["paired_comparisons"]))
print("mT5 annotated variant:", results["models"]["mt5-large"]["human_prediction_variant"])
